# Typhoon Dataset Merge Notebook

## What this notebook does

1. Standardizes merge keys (`Year`, `Typhoon`, `Region`)
2. Diagnoses overlap and missing combinations
3. Aggregates duplicated info+infra keys to avoid many-to-many merge inflation
4. Performs a full outer merge to preserve information
5. Produces two analysis-ready outputs:
   - Matched-only dataset (`both`)
   - Union dataset (all rows) with missingness/source flags


In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

## 1) Locate and Load Source Files

This cell supports running the notebook from either project root or the `notebooks` folder.


In [2]:
cwd = Path.cwd()
if (cwd / 'data' / 'merged').exists():
    project_root = cwd
elif (cwd.parent / 'data' / 'merged').exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError('Could not find data/merged from current notebook working directory.')

data_dir = project_root / 'data' / 'merged'
info_path = data_dir / 'typhoon-info-infra-project.csv'
impact_path = data_dir / 'cleaned_typhoon_impacts.csv'

info_raw = pd.read_csv(info_path)
impact_raw = pd.read_csv(impact_path)

print(f'Project root: {project_root}')
print(f'Info+Infra shape: {info_raw.shape}')
print(f'Impact shape: {impact_raw.shape}')

Project root: c:\Users\Rainer Gonzaga\Documents\GitHub\DLSU\second-year\term-2\flood-control-health-outcomes
Info+Infra shape: (398, 11)
Impact shape: (386, 12)


In [3]:
print('INFO+INFRA columns:')
print(info_raw.columns.tolist())
print('\nIMPACT columns:')
print(impact_raw.columns.tolist())

INFO+INFRA columns:
['Typhoon', 'Year', 'Location', 'Max 24-hour Rainfall (mm)', 'Peak Gust (10 mins sustained) (m/s)', 'Date', 'Province', 'Region', 'Cumulative_Budget_To_Date', 'Cumulative_Variance_To_Date', 'Variance_Ratio_To_Date']

IMPACT columns:
['Cyclone Name', 'Region', 'Year', 'Category', 'Deaths', 'Injuries', 'Affected', 'Houses destroyed', 'Houses damaged', 'Total Houses', 'Damage to Infrastructure (PhP)', 'Damage to Agriculture + Fisheries (PhP)']


## 2) Standardize Join Keys

We use a common key:

- `Year`
- `typhoon_key` (normalized typhoon name)
- `region_key` (normalized region text)

Normalization rules: uppercase, trim spaces, remove punctuation/non-alphanumeric symbols, collapse repeated spaces.


In [4]:
def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).upper().strip()
    text = re.sub(r'[^A-Z0-9]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else pd.NA

# Optional alias mapping for known typhoon name variants or aliases.
# Example: {'ULYSSES': 'VAMCO'} if you decide to unify international/local names.
TYPHOON_ALIAS_MAP = {}

info = info_raw.rename(columns={'Typhoon': 'typhoon_name'}).copy()
impact = impact_raw.rename(columns={'Cyclone Name': 'typhoon_name'}).copy()

for df in (info, impact):
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
    df['typhoon_key'] = df['typhoon_name'].map(normalize_text).replace(TYPHOON_ALIAS_MAP)
    df['region_key'] = df['Region'].map(normalize_text)

key_cols = ['Year', 'typhoon_key', 'region_key']

print('Missing keys in info+infra:', info[key_cols].isna().any(axis=1).sum())
print('Missing keys in impact:', impact[key_cols].isna().any(axis=1).sum())

Missing keys in info+infra: 0
Missing keys in impact: 0


## 3) Diagnose Coverage and Mismatch

This gives a transparent view of how many key combinations overlap and how many are exclusive to each dataset.


In [5]:
info_keys = info[key_cols].dropna().drop_duplicates()
impact_keys = impact[key_cols].dropna().drop_duplicates()

info_key_set = set(map(tuple, info_keys.to_numpy()))
impact_key_set = set(map(tuple, impact_keys.to_numpy()))

intersection = info_key_set & impact_key_set
only_info = info_key_set - impact_key_set
only_impact = impact_key_set - info_key_set

coverage_summary = pd.DataFrame({
    'metric': ['info_unique_keys', 'impact_unique_keys', 'intersection', 'only_info', 'only_impact'],
    'value': [len(info_key_set), len(impact_key_set), len(intersection), len(only_info), len(only_impact)]
})
coverage_summary

,metric,value
0,info_unique_keys,200
1,impact_unique_keys,386
2,intersection,81
3,only_info,119
4,only_impact,305


In [6]:
only_info_sample = pd.DataFrame(list(only_info), columns=key_cols).sort_values(key_cols).head(10)
only_impact_sample = pd.DataFrame(list(only_impact), columns=key_cols).sort_values(key_cols).head(10)

print('Sample keys only in info+infra:')
display(only_info_sample)
print('Sample keys only in impact:')
display(only_impact_sample)

Sample keys only in info+infra:


,Year,typhoon_key,region_key
58,2019,FALCON,REGION IV B
95,2019,FALCON,REGION V
47,2019,HANNA,REGION III
57,2019,HANNA,REGION IV B
117,2019,INENG,REGION I
115,2019,INENG,REGION II
89,2019,KABAYAN,NCR
73,2019,KABAYAN,REGION I
87,2019,KABAYAN,REGION IV A
69,2019,NIMFA,REGION III


Sample keys only in impact:


,Year,typhoon_key,region_key
110,2019,AMANG,REGION XIII
211,2019,CHEDENG,REGION XI
58,2019,FALCON,CAR
17,2019,FALCON,REGION II
234,2019,MARILYN,REGION II
235,2019,MARILYN,REGION III
143,2019,MARILYN,REGION IX
79,2019,MARILYN,REGION VI
116,2019,MARILYN,REGION XI
46,2019,MARILYN,REGION XII


## 4) Handle Duplicates Before Merge

`info+infra` can contain multiple rows per key, so we aggregate first to avoid row explosion in merge results.

Aggregation logic:

- Numeric columns: sum by default
- `Variance_Ratio_To_Date`: mean (ratio-style metric)
- Descriptive fields: keep first non-null representative value
- `infra_project_count`: number of contributing rows per key


In [7]:
info_dup_groups = (
    info.groupby(key_cols, dropna=False)
    .size()
    .reset_index(name='count')
    .query('count > 1')
    .sort_values('count', ascending=False)
)

print(f'Duplicate key groups in info+infra: {len(info_dup_groups)}')
display(info_dup_groups.head(10))

Duplicate key groups in info+infra: 102


,Year,typhoon_key,region_key,count
154,2024,OFEL,REGION II,6
100,2022,PAENG,REGION IV A,6
88,2022,AGATON,REGION VIII,5
103,2023,AMANG,REGION V,5
59,2021,BISING,REGION VIII,5
45,2020,ROLLY,REGION V,5
91,2022,FLORITA,REGION II,5
167,2025,ISANG,REGION III,5
183,2025,RAMIL,REGION III,5
181,2025,PAOLO,REGION III,5


In [8]:
numeric_cols = info.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Year']

agg_map = {col: 'sum' for col in numeric_cols}
if 'Variance_Ratio_To_Date' in agg_map:
    agg_map['Variance_Ratio_To_Date'] = 'mean'

for col in ['typhoon_name', 'Region', 'Date', 'Province', 'Location']:
    if col in info.columns:
        agg_map[col] = 'first'

info_agg = info.groupby(key_cols, dropna=False, as_index=False).agg(agg_map)
info_agg['infra_project_count'] = info.groupby(key_cols, dropna=False).size().values

print('Info+infra shape before aggregation:', info.shape)
print('Info+infra shape after aggregation:', info_agg.shape)

Info+infra shape before aggregation: (398, 13)
Info+infra shape after aggregation: (200, 14)


## 5) Full Outer Merge (Information-Preserving)

Why full outer?

- Keeps all keys from both datasets
- Prevents accidental loss from non-overlapping typhoon-region-year combinations
- Adds `_merge` to show row provenance (`left_only`, `right_only`, `both`)


In [9]:
merged_union = info_agg.merge(
    impact,
    on=key_cols,
    how='outer',
    suffixes=('_info_infra', '_impact'),
    indicator=True
)

merged_union['merge_source'] = merged_union['_merge'].map({
    'left_only': 'info_infra_only',
    'right_only': 'impact_only',
    'both': 'both'
})

# Explicit missingness flags (important for downstream analysis).
merged_union['has_infra_weather_data'] = merged_union['merge_source'].isin(['info_infra_only', 'both'])
merged_union['has_impact_data'] = merged_union['merge_source'].isin(['impact_only', 'both'])

merge_distribution = merged_union['merge_source'].value_counts(dropna=False).rename_axis('merge_source').reset_index(name='rows')
merge_distribution

,merge_source,rows
0,impact_only,305
1,info_infra_only,119
2,both,81


## 6) Produce Two Analysis Datasets

- **Union dataset**: full outer result, best for descriptive coverage and auditability
- **Matched dataset**: rows where both sides exist, best for strict inferential models requiring both feature groups


In [10]:
merged_matched = merged_union.loc[merged_union['merge_source'] == 'both'].copy()

print('Union rows:', len(merged_union))
print('Matched rows:', len(merged_matched))
print('Matched proportion:', round(len(merged_matched) / len(merged_union), 4) if len(merged_union) else np.nan)

Union rows: 505
Matched rows: 81
Matched proportion: 0.1604


## 7) Save Outputs

This writes reproducible outputs to `data/merged` for your team workflow.


In [11]:
union_path = data_dir / 'typhoon_union_full_outer.csv'
matched_path = data_dir / 'typhoon_matched_inner.csv'
quality_path = data_dir / 'merge_quality_report.csv'

merged_union.to_csv(union_path, index=False)
merged_matched.to_csv(matched_path, index=False)

quality_report = pd.concat([
    coverage_summary.assign(section='coverage'),
    merge_distribution.rename(columns={'merge_source': 'metric', 'rows': 'value'}).assign(section='merge_source_distribution')
], ignore_index=True)

quality_report.to_csv(quality_path, index=False)

print('Saved:')
print('-', union_path)
print('-', matched_path)
print('-', quality_path)

Saved:
- c:\Users\Rainer Gonzaga\Documents\GitHub\DLSU\second-year\term-2\flood-control-health-outcomes\data\merged\typhoon_union_full_outer.csv
- c:\Users\Rainer Gonzaga\Documents\GitHub\DLSU\second-year\term-2\flood-control-health-outcomes\data\merged\typhoon_matched_inner.csv
- c:\Users\Rainer Gonzaga\Documents\GitHub\DLSU\second-year\term-2\flood-control-health-outcomes\data\merged\merge_quality_report.csv


## 8) Interpretation Notes

- Use `typhoon_union_full_outer.csv` when you need complete coverage and transparent missingness.
- Use `typhoon_matched_inner.csv` for models/tests requiring both infrastructure/weather and impact variables present.
- Avoid replacing missing impact values with 0 unless domain logic guarantees true zero.
- If additional typhoon aliases are identified, update `TYPHOON_ALIAS_MAP` and rerun from the key-standardization cell.
